# 03 KIỂM THỬ HÀNH ĐỘNG & TÁC TỬ RANDOM (ACTION & RANDOM AGENT TESTING)

Notebook này thực hiện:
1. Kiểm tra khả năng sinh nước đi hợp lệ từ trạng thái bàn cờ ban đầu (`create_initial_state`).
2. Kiểm tra các bất biến của `RandomAgent` (tính hợp lệ, tính tái lập theo `seed`, đo thời gian suy nghĩ).
3. Mô phỏng ván đấu mẫu giữa **Random Đỏ** và **Random Xanh** với bảng nhật ký nước đi chi tiết.
4. Chạy thực nghiệm batch 10 ván đấu để thống kê kết quả, số lượt và hiệu năng của engine/agent.

In [1]:
from pathlib import Path
import sys
import time
import pandas as pd
from collections import Counter

# 1. Cấu hình đường dẫn nạp module từ repo
cwd = Path.cwd().resolve()
PROJECT_ROOT = next((p for p in [cwd, *cwd.parents] if (p / "packages").is_dir() and (p / "research").is_dir()), None)
if PROJECT_ROOT is None:
    raise RuntimeError("Hãy mở notebook từ thư mục repo Commander-Chess-AI")

if str(PROJECT_ROOT / "packages") not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT / "packages"))

# 2. Nạp Game Engine và Agent
from commander_game_engine.board import Board, Terrain, Position
from commander_game_engine.game_state import GameState, Player, PieceType, GameStatus
from commander_game_engine.initial_setup import create_initial_state, INITIAL_POSITIONS
from commander_game_engine.move_generator import generate_legal_actions
from commander_game_engine.engine import apply_action, EngineResult
from agents import RandomAgent, AgentDecision, NoLegalActionsError

print("Nạp các module Game Engine & Agents thành công!")

Nạp các module Game Engine & Agents thành công!


## 1. Kiểm tra bàn cờ ban đầu & Sinh nước đi mở màn (Opening Legal Actions)

In [2]:
state = create_initial_state()
print(f"Trạng thái khởi tạo: {len(state.pieces)} quân cờ trên bàn.")
print(f"Phe đi trước: {state.current_player.value.upper()}")

# Sinh nước đi mở màn của phe Đỏ
start_t = time.perf_counter()
opening_actions = generate_legal_actions(state)
gen_time_ms = (time.perf_counter() - start_t) * 1000

print(f"Số lượng nước đi hợp lệ ở lượt 1: {len(opening_actions)} nước (sinh trong {gen_time_ms:.2f} ms)")

# Phân loại nước đi theo loại quân cờ
type_counts = Counter()
for act in opening_actions:
    piece = state.pieces[act.piece_id]
    type_counts[piece.piece_type.value] += 1

df_types = pd.DataFrame(list(type_counts.items()), columns=["Loại quân", "Số nước đi khả dĩ"]).sort_values(by="Số nước đi khả dĩ", ascending=False)
display(df_types)

assert len(opening_actions) > 0, "Lượt mở màn phải có nước đi hợp lệ!"
print("PASS: Khởi tạo và sinh nước đi mở màn thành công.")

Trạng thái khởi tạo: 38 quân cờ trên bàn.
Phe đi trước: RED
Số lượng nước đi hợp lệ ở lượt 1: 81 nước (sinh trong 9.59 ms)
       Loại quân  Số nước đi khả dĩ
8           navy                 22
2      artillery                 15
0      commander                  9
4           tank                  8
1        missile                  6
3  anti_aircraft                  6
6       engineer                  6
7        militia                  6
5       infantry                  3
PASS: Khởi tạo và sinh nước đi mở màn thành công.


## 2. Kiểm thử tính đúng đắn của RandomAgent

In [3]:
# 1. Kiểm tra chọn nước đi và metadata
agent_a = RandomAgent(name="Random_A", seed=42)
decision_a = agent_a.select_action(state, opening_actions)

print(f"Agent đã chọn: {decision_a.action.piece_id} [{decision_a.action.action_type.value}] {decision_a.action.start} -> {decision_a.action.end}")
print("Metadata trả về:", decision_a.metadata)
assert decision_a.action in opening_actions, "Nước đi phải thuộc danh sách hợp lệ!"

# 2. Kiểm tra tính tái lập qua seed (Reproducibility)
agent_b = RandomAgent(name="Random_B", seed=42)
decision_b = agent_b.select_action(state, opening_actions)
assert decision_a.action == decision_b.action, "Hai agent cùng seed phải đưa ra cùng nước đi!"
print("PASS: Tính tái lập tuyệt đối qua seed (Reproducibility) đạt chuẩn.")

# 3. Kiểm tra xử lý ngoại lệ khi danh sách nước đi rỗng
try:
    agent_a.select_action(state, [])
    assert False, "Không phát hiện lỗi khi legal_actions rỗng"
except NoLegalActionsError as e:
    print(f"PASS: Xử lý ngoại lệ chuẩn khi danh sách rỗng -> {e}")

Agent đã chọn: red_missile_1 [move] (2, 6) -> (3, 6)
Metadata trả về: {'agent_name': 'Random_A', 'algorithm': 'random', 'version': '1.0.0', 'seed': 42, 'decision_time_ms': 0.0044000043999403715, 'candidates_count': 81}
PASS: Tính tái lập tuyệt đối qua seed (Reproducibility) đạt chuẩn.
PASS: Xử lý ngoại lệ chuẩn khi danh sách rỗng -> Agent 'Random_A' không thể chọn nước đi vì legal_actions rỗng.


## 3. Mô phỏng 1 ván đấu hoàn chỉnh: Random Đỏ vs Random Xanh

In [4]:
# Khởi tạo trận đấu
match_state = create_initial_state()
agent_red = RandomAgent(name="Red_Bot", seed=101)
agent_blue = RandomAgent(name="Blue_Bot", seed=202)
agents = {Player.RED: agent_red, Player.BLUE: agent_blue}

MAX_TURNS = 120  # Giới hạn số lượt để tránh lặp vô tận
history = []
winner = None
termination_reason = "Max turns reached (Draw)"

print(f"Bắt đầu ván đấu (tối đa {MAX_TURNS} lượt)...")
t_match_start = time.perf_counter()

for turn in range(1, MAX_TURNS + 1):
    current_player = match_state.current_player
    active_agent = agents[current_player]
    
    actions = generate_legal_actions(match_state)
    if not actions:
        termination_reason = f"Phe {current_player.value.upper()} hết nước đi hợp lệ"
        break
        
    decision = active_agent.select_action(match_state, actions)
    action = decision.action
    
    result = apply_action(match_state, action)
    assert result.accepted, f"Hành động bị engine từ chối: {result.verdict.reason}"
    
    # Ghi lại sự kiện của lượt
    event_desc = ", ".join(f"{e.event_type.value}: {e.detail}" for e in result.events if e.detail)
    history.append({
        "Lượt": turn,
        "Phe": current_player.value.upper(),
        "Quân": action.piece_id,
        "Hành động": action.action_type.value,
        "Từ": str(action.start),
        "Đến": str(action.end),
        "Thời gian (ms)": round(decision.metadata["decision_time_ms"], 3),
        "Sự kiện": event_desc
    })
    
    match_state = result.state
    if result.terminal:
        winner = result.winner.value.upper() if result.winner else "None"
        termination_reason = f"Tư lệnh bị tiêu diệt! Phe {winner} chiến thắng!"
        break

total_match_time = time.perf_counter() - t_match_start
print(f"\n=== KẾT THÚC VÁN ĐẤU ===")
print(f"Lý do kết thúc: {termination_reason}")
print(f"Tổng số lượt: {len(history)} lượt | Thời gian chạy: {total_match_time:.2f} giây")

# Hiển thị 20 lượt đầu tiên và 10 lượt cuối cùng
df_history = pd.DataFrame(history)
print("\n--- 15 LƯỢT ĐẦU TIÊN ---")
display(df_history.head(15))
if len(df_history) > 15:
    print("\n--- 10 LƯỢT CUỐI CÙNG ---")
    display(df_history.tail(10))

Bắt đầu ván đấu (tối đa 120 lượt)...

=== KẾT THÚC VÁN ĐẤU ===
Lý do kết thúc: Max turns reached (Draw)
Tổng số lượt: 120 lượt | Thời gian chạy: 1.16 giây

--- 15 LƯỢT ĐẦU TIÊN ---
    Lượt  ...                                            Sự kiện
0      1  ...  moved: Tới tọa độ (2, 2)., turn_changed: Đến l...
1      2  ...  moved: Tới tọa độ (8, 3)., turn_changed: Đến l...
2      3  ...  moved: Tới tọa độ (1, 10)., turn_changed: Đến ...
3      4  ...  moved: Tới tọa độ (9, 1)., turn_changed: Đến l...
4      5  ...  moved: Tới tọa độ (1, 2)., turn_changed: Đến l...
5      6  ...  moved: Tới tọa độ (6, 7)., turn_changed: Đến l...
6      7  ...  moved: Tới tọa độ (5, 10)., turn_changed: Đến ...
7      8  ...  moved: Tới tọa độ (9, 8)., turn_changed: Đến l...
8      9  ...  moved: Tới tọa độ (5, 6)., turn_changed: Đến l...
9     10  ...  moved: Tới tọa độ (10, 9)., turn_changed: Đến ...
10    11  ...  moved: Tới tọa độ (0, 9)., turn_changed: Đến l...
11    12  ...  captured: Bị ăn quân., p

## 4. Thực nghiệm Batch: Đánh giá 10 ván đấu Random vs Random

In [5]:
NUM_GAMES = 10
MAX_TURNS_BATCH = 150
batch_results = []

print(f"Đang chạy thực nghiệm {NUM_GAMES} ván Random vs Random...")
t_batch_start = time.perf_counter()

for game_idx in range(1, NUM_GAMES + 1):
    g_state = create_initial_state()
    bot_r = RandomAgent(name=f"Red_{game_idx}", seed=1000 + game_idx)
    bot_b = RandomAgent(name=f"Blue_{game_idx}", seed=2000 + game_idx)
    g_agents = {Player.RED: bot_r, Player.BLUE: bot_b}
    
    g_winner = "Hòa (Max turns)"
    turns_played = 0
    
    for t in range(1, MAX_TURNS_BATCH + 1):
        p = g_state.current_player
        acts = generate_legal_actions(g_state)
        if not acts:
            g_winner = f"Thắng do đối thủ hết nước ({('BLUE' if p == Player.RED else 'RED')})"
            turns_played = t
            break
            
        dec = g_agents[p].select_action(g_state, acts)
        res = apply_action(g_state, dec.action)
        g_state = res.state
        turns_played = t
        
        if res.terminal:
            g_winner = f"Phe {res.winner.value.upper()} thắng (Diệt Tư lệnh)"
            break
            
    batch_results.append({
        "Ván": game_idx,
        "Seed Đỏ": 1000 + game_idx,
        "Seed Xanh": 2000 + game_idx,
        "Số lượt": turns_played,
        "Kết quả": g_winner,
        "Số quân còn lại": len(g_state.pieces)
    })

t_batch_total = time.perf_counter() - t_batch_start
df_batch = pd.DataFrame(batch_results)
display(df_batch)

print(f"\nTổng thời gian chạy {NUM_GAMES} ván: {t_batch_total:.2f} giây (Trung bình {t_batch_total/NUM_GAMES:.2f}s / ván)")
print("Phân bố kết quả:")
display(df_batch["Kết quả"].value_counts().to_frame("Số ván"))

Đang chạy thực nghiệm 10 ván Random vs Random...
   Ván  Seed Đỏ  ...                       Kết quả  Số quân còn lại
0    1     1001  ...               Hòa (Max turns)               31
1    2     1002  ...               Hòa (Max turns)               32
2    3     1003  ...               Hòa (Max turns)               32
3    4     1004  ...               Hòa (Max turns)               34
4    5     1005  ...               Hòa (Max turns)               32
5    6     1006  ...               Hòa (Max turns)               31
6    7     1007  ...               Hòa (Max turns)               34
7    8     1008  ...               Hòa (Max turns)               31
8    9     1009  ...  Phe RED thắng (Diệt Tư lệnh)               30
9   10     1010  ...               Hòa (Max turns)               31

[10 rows x 6 columns]

Tổng thời gian chạy 10 ván: 14.14 giây (Trung bình 1.41s / ván)
Phân bố kết quả:
                              Số ván
Kết quả                             
Hòa (Max turns)         